In [ ]:
# Databricks notebook source
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import os

# ---------------------------------------------------------------------------
# Configuracion externalizada (widgets de Databricks / variables de entorno).
# ---------------------------------------------------------------------------
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "landing")
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("export_prefix", "gold/web_export")


def _cfg(name: str, default: str = "") -> str:
    """Resuelve un parametro: widget -> variable de entorno -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


STORAGE_ACCOUNT = _cfg("storage_account")
CONTAINER = _cfg("container", "landing")
CATALOG = _cfg("catalog", "job_offers")
GOLD_SCHEMA = _cfg("gold_schema", "gold")
EXPORT_PREFIX = _cfg("export_prefix", "gold/web_export")

if not STORAGE_ACCOUNT:
    raise ValueError(
        "Falta el parametro obligatorio 'storage_account': indica la cuenta "
        "de almacenamiento del contenedor del export.")


# Export web (Gold -> Parquet + meta.json)

Genera el export que consume el dashboard web estatico (`docs/dashboard`):
lee las tablas Gold, las proyecta al contrato publico (columnas exactas y
geografia portada de Power BI) y escribe un Parquet por tabla mas `meta.json`
(frescura, recuentos, tamano y modo) en ADLS (`abfss://`).

Se ejecuta al final del job diario, despues de `gold_build`. No escribe datos
crudos, descripciones ni columnas internas: solo campos publicos. La
credencial la aporta el cluster (no hay SAS ni tokens en el notebook).


In [ ]:
# Tablas Gold ya construidas: el notebook solo proyecta (no deduplica).
fact_offers = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_offers")
fact_offer_skills = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_offer_skills")
dim_skill_list = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_skill_list")
dim_calendar = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_calendar")


In [ ]:
# Modulo hermano del repo (mismo patron que Prepare_Gold en gold_build.ipynb).
import Prepare_Web_Export as web_export

dest = (f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
        f"{EXPORT_PREFIX.strip('/')}")

tables = web_export.build_web_export(
    spark, fact_offers, fact_offer_skills, dim_skill_list, dim_calendar)

table_counts, sources, data_date = web_export.collect_export_stats(tables)
meta = web_export.build_meta(table_counts, sources, data_date)

# size_provider se llama dentro de write_web_export despues de escribir los
# cuatro Parquet y antes de meta.json. Se filtran solo los .parquet: en las
# reejecuciones el meta.json anterior seguiria en dest y no debe contar.
size_provider = lambda: sum(
    f.size for f in dbutils.fs.ls(dest) if f.name.endswith(".parquet"))

updated_meta = web_export.write_web_export(
    spark, tables, dest, meta=meta, size_provider=size_provider)


In [ ]:
size_mb = round(updated_meta["size_bytes"] / 1024 / 1024, 2)

print("Export web: resumen")
print(f"  destino:     {dest}")
print(f"  modo:        {updated_meta['mode']}")
print(f"  tamano:      {updated_meta['size_bytes']} bytes ({size_mb} MB)")
print(f"  fecha datos: {updated_meta['data_date']}")
print(f"  generado:    {updated_meta['generated_at']}")
print(f"  tablas:      {updated_meta['tables']}")
print(f"  fuentes:     {updated_meta['sources']}")

display(spark.createDataFrame(
    [("dest", dest),
     ("mode", updated_meta["mode"]),
     ("size_bytes", str(updated_meta["size_bytes"])),
     ("size_mb", str(size_mb)),
     ("data_date", str(updated_meta["data_date"])),
     ("generated_at", updated_meta["generated_at"])],
    ["campo", "valor"]))
